# Phase 11: Risk Scoring (0-30 Low, 30-70 Medium, 70-100 High)

Turns the champion LightGBM model's raw fraud probability into a single, business-friendly **Risk Score from 0 to 100**, bucketed into **Low / Medium / High**.

### Why this isn't just `probability * 100`
A quick look at the model's actual output on the test set shows why a naive `probability * 100` mapping would be useless here: because Phase 4's oversampling only rebalances the *training* set, the model's raw probabilities on real (imbalanced) data are extremely compressed near zero — 99.7% of test transactions score below probability 0.01, and the 75th percentile is around 0.00001. A straight linear scaling would put almost every transaction at Risk Score ≈ 0, with no usable separation between "obviously fine" and "somewhat suspicious."

Instead, this notebook builds a **piecewise-linear mapping anchored to two real decision thresholds** from Phase 9's threshold analysis, so the 0-30 / 30-70 / 70-100 bands actually mean something operationally:
- **0 → 30 (Low)**: probability below the "catch 90% of fraud" threshold — the model has no meaningful concern.
- **30 → 70 (Medium)**: probability between that threshold and the cost-optimal flag threshold — some signal, worth a second look but not enough to auto-block.
- **70 → 100 (High)**: probability at or above the cost-optimal flag threshold — this is exactly what the model would auto-flag as fraud.

This way "High risk" isn't an arbitrary label — it's precisely the set of transactions the Phase 9 threshold decision would act on, and "Medium" is a genuine early-warning zone below it.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

## Phase 11, Step 1: Setup — Load Model & Data, Determine the Two Anchor Thresholds

Tries to load `threshold_config.pkl` from Phase 9 first (the real, business-calibrated thresholds). If it isn't there yet, this recomputes the same two thresholds on the fly from the validation set, using the identical logic as notebook 06, so this notebook works standalone either way — but running notebook 06 first is still recommended for the full threshold analysis and write-up.

- **`T_LOW`**: the highest threshold that still catches ≥90% of fraud on the validation set (the "High-recall" candidate from Phase 9) — the Low/Medium boundary.
- **`T_HIGH`**: the cost-optimal threshold from Phase 9 — the Medium/High boundary, and the model's actual flagging cutoff.

In [ ]:
X_val = pd.read_csv('../data/processed/X_val.csv')
y_val = pd.read_csv('../data/processed/y_val.csv').squeeze()
X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

best_supervised_model = joblib.load('../models/best_supervised_model.pkl')
y_proba_val = best_supervised_model.predict_proba(X_val)[:, 1]

try:
    threshold_config = joblib.load('../models/threshold_config.pkl')
    T_HIGH = threshold_config['final_threshold']
    # Pull the high-recall candidate out of Phase 9's saved candidates dict
    recall_key = next((k for k in threshold_config['candidates'] if 'High-recall' in k), None)
    T_LOW = threshold_config['candidates'][recall_key] if recall_key else T_HIGH * 0.25
    print(f"Loaded Phase 9 threshold_config.pkl -> T_LOW={T_LOW}, T_HIGH={T_HIGH} (final_threshold)")
except FileNotFoundError:
    print("threshold_config.pkl not found (run notebook 06 first for the full analysis) "
          "-> recomputing the two anchor thresholds directly from the validation set.")
    thresholds = np.arange(0.01, 1.00, 0.01)
    rec_list = [recall_score(y_val, (y_proba_val >= t).astype(int), zero_division=0) for t in thresholds]
    f1_list = [f1_score(y_val, (y_proba_val >= t).astype(int), zero_division=0) for t in thresholds]

    recall_ok = thresholds[np.array(rec_list) >= 0.90]
    T_LOW = float(recall_ok.max()) if len(recall_ok) > 0 else float(thresholds[0])
    T_HIGH = float(thresholds[np.argmax(f1_list)])  # F1-optimal as a stand-in for cost-optimal
    print(f"Computed on the fly -> T_LOW={T_LOW} (>=90% recall), T_HIGH={T_HIGH} (F1-optimal, "
          "stand-in for cost-optimal since raw Amount recovery isn't needed for this notebook)")

assert T_LOW < T_HIGH, "T_LOW must be below T_HIGH for the piecewise mapping to make sense"
print(f"\nFinal anchors: T_LOW={T_LOW}, T_HIGH={T_HIGH}")

## Phase 11, Step 2: The Risk Score Function

A monotonic, piecewise-linear map from probability → 0-100:

```
p < T_LOW:          score = (p / T_LOW) * 30                        -> [0, 30)
T_LOW <= p < T_HIGH: score = 30 + (p - T_LOW)/(T_HIGH - T_LOW) * 40  -> [30, 70)
p >= T_HIGH:         score = 70 + (p - T_HIGH)/(1 - T_HIGH) * 30     -> [70, 100]
```

Each segment is linear on its own range, so scores within a band still rank-order transactions correctly (a Risk Score of 45 is still riskier than 35) — only the *scale* is stretched to make the compressed low-probability region usable.

In [ ]:
def risk_score(proba, t_low=T_LOW, t_high=T_HIGH):
    """Map a fraud probability (or array of them) to a 0-100 Risk Score."""
    p = np.asarray(proba, dtype=float)
    score = np.zeros_like(p)

    low_mask = p < t_low
    mid_mask = (p >= t_low) & (p < t_high)
    high_mask = p >= t_high

    score[low_mask] = (p[low_mask] / t_low) * 30
    score[mid_mask] = 30 + (p[mid_mask] - t_low) / (t_high - t_low) * 40
    score[high_mask] = 70 + np.clip((p[high_mask] - t_high) / (1 - t_high), 0, 1) * 30

    return np.clip(score, 0, 100)

def risk_band(score):
    score = np.asarray(score, dtype=float)
    return np.select(
        [score < 30, score < 70],
        ['Low', 'Medium'],
        default='High'
    )

# Quick sanity check
test_probs = np.array([0.0, T_LOW / 2, T_LOW, (T_LOW + T_HIGH) / 2, T_HIGH, (T_HIGH + 1) / 2, 1.0])
for p in test_probs:
    s = risk_score(np.array([p]))[0]
    print(f"probability={p:.5f} -> Risk Score={s:6.2f} -> Band={risk_band(np.array([s]))[0]}")

## Phase 11, Step 3: Apply to the Test Set & Evaluate the Bands

The real question for a risk-tiering scheme isn't accuracy in the abstract — it's *does the High band actually concentrate the fraud, and is the Low band actually safe to leave alone?*

In [ ]:
y_proba_test = best_supervised_model.predict_proba(X_test)[:, 1]
scores_test = risk_score(y_proba_test)
bands_test = risk_band(scores_test)

results_df = pd.DataFrame({
    'probability': y_proba_test,
    'risk_score': scores_test,
    'risk_band': pd.Categorical(bands_test, categories=['Low', 'Medium', 'High'], ordered=True),
    'actual_fraud': y_test.values
})

band_summary = results_df.groupby('risk_band', observed=True).agg(
    transactions=('actual_fraud', 'count'),
    actual_fraud_count=('actual_fraud', 'sum'),
    fraud_rate=('actual_fraud', 'mean')
).round(4)
band_summary['pct_of_all_fraud_caught'] = (
    band_summary['actual_fraud_count'] / band_summary['actual_fraud_count'].sum() * 100
).round(1)

print("=== Risk Band Summary — Test Set ===\n")
print(band_summary)

total_fraud = int(results_df['actual_fraud'].sum())
high_fraud = int(band_summary.loc['High', 'actual_fraud_count'])
medhigh_fraud = int(band_summary.loc[['Medium', 'High'], 'actual_fraud_count'].sum())
print(f"\n{high_fraud}/{total_fraud} ({high_fraud/total_fraud:.1%}) of all fraud lands in the High band alone.")
print(f"{medhigh_fraud}/{total_fraud} ({medhigh_fraud/total_fraud:.1%}) of all fraud lands in Medium+High combined.")
print(f"Low band fraud rate: {band_summary.loc['Low', 'fraud_rate']:.4%} "
      "(essentially safe to leave un-reviewed).")

## Phase 11, Step 4: Visualize the Score Distribution & Band Quality

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(results_df.loc[results_df['actual_fraud'] == 0, 'risk_score'], bins=50,
             alpha=0.6, label='Legit', color='steelblue', log=True)
axes[0].hist(results_df.loc[results_df['actual_fraud'] == 1, 'risk_score'], bins=50,
             alpha=0.7, label='Fraud', color='crimson', log=True)
axes[0].axvline(30, linestyle='--', color='grey')
axes[0].axvline(70, linestyle='--', color='grey')
axes[0].set_xlabel('Risk Score (0-100)'); axes[0].set_ylabel('Count (log scale)')
axes[0].set_title('Risk Score Distribution — Test Set')
axes[0].legend()

band_summary['fraud_rate'].plot(kind='bar', ax=axes[1], color=['seagreen', 'orange', 'crimson'])
axes[1].set_ylabel('Fraud Rate Within Band')
axes[1].set_title('Fraud Concentration by Risk Band')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## Phase 11, Step 5: Save Artifacts for Phase 12 (FastAPI) Onward

Saves the two anchor thresholds and band boundaries so the FastAPI endpoint (Phase 12) can call `risk_score()` and `risk_band()` on a single incoming transaction without needing the validation set or Phase 9's notebook — just this config and the model.

In [ ]:
risk_scoring_config = {
    't_low': float(T_LOW),
    't_high': float(T_HIGH),
    'band_boundaries': {'low_max': 30, 'medium_max': 70},
    'band_definitions': {
        'Low': 'probability < t_low — no meaningful fraud signal',
        'Medium': 't_low <= probability < t_high — worth a second look, not auto-blocked',
        'High': 'probability >= t_high — matches the model\'s actual auto-flag threshold'
    }
}
joblib.dump(risk_scoring_config, '../models/risk_scoring_config.pkl')

results_df.head(200).to_csv('../models/risk_scored_test_sample.csv', index=False)

print("Saved models/risk_scoring_config.pkl  (t_low, t_high, band boundaries)")
print("Saved models/risk_scored_test_sample.csv  (first 200 scored test transactions, for reference)")
print("\nPhase 11 complete.")
print(f"T_LOW={T_LOW}, T_HIGH={T_HIGH}")
print("Ready for Phase 12: FastAPI (POST /predict endpoint).")